In [0]:
%run /Workspace/de_shared/config



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_silver = (
    spark.read
    .format("delta")
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://medinsight-partd-yash/delta/silver/prescriber_claims/")
)

print(f"Silver row count: {df_silver.count():,}")

Silver row count: 23,850,038


In [0]:
# Aggregate by state, specialty, drug, year
df_agg = df_silver.groupBy(
    "prescriber_npi",
    "prescriber_last_name",
    "prescriber_first_name",
    "state",
    "specialty",
    "drug_brand",
    "drug_generic",
    "data_year"
).agg(
    sum("total_claims").alias("total_claims"),
    sum("total_cost_usd").alias("total_cost_usd"),
    sum("total_day_supply").alias("total_day_supply")
)

# Add prescriber rank within each state + specialty + drug + year group
window_spec = Window.partitionBy(
    "state", "specialty", "drug_generic", "data_year"
).orderBy(col("total_claims").desc())

df_gold = df_agg \
    .withColumn("prescriber_rank", rank().over(window_spec)) \
    .withColumn("_updated_at", current_timestamp())

print(f"Gold row count: {df_gold.count():,}")

Gold row count: 23,850,038


In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .partitionBy("data_year") \
    .save("s3a://medinsight-partd-yash/delta/gold/territory_scorecard/")

print("Gold write complete")

Gold write complete


In [0]:
df_verify = (
    spark.read
    .format("delta")
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://medinsight-partd-yash/delta/gold/territory_scorecard/")
)

print(f"Gold row count: {df_verify.count():,}")

# Show rank 1 prescribers — top doctor per state+specialty+drug
df_verify.filter(col("prescriber_rank") == 1) \
    .select(
        "state", "specialty", "drug_generic",
        "prescriber_npi", "prescriber_last_name",
        "total_claims", "total_cost_usd", "prescriber_rank"
    ) \
    .orderBy(col("total_claims").desc()) \
    .show(10, truncate=False)

Gold row count: 23,850,038
+-----+-----------------------+----------------------------+--------------+--------------------+------------+--------------+---------------+
|state|specialty              |drug_generic                |prescriber_npi|prescriber_last_name|total_claims|total_cost_usd|prescriber_rank|
+-----+-----------------------+----------------------------+--------------+--------------------+------------+--------------+---------------+
|CA   |Emergency Medicine     |Varicella-Zoster Ge/As01b/Pf|1639279417    |Azad                |173786      |3.231570666E7 |1              |
|FL   |Family Practice        |Varicella-Zoster Ge/As01b/Pf|1356534994    |Davis               |152819      |2.865186048E7 |1              |
|TX   |Radiation Oncology     |Varicella-Zoster Ge/As01b/Pf|1538169388    |Han                 |95427       |1.784212986E7 |1              |
|MA   |Otolaryngology         |Varicella-Zoster Ge/As01b/Pf|1265429617    |Bogdasarian         |80094       |1.488226166E7 |1  

In [0]:
assert df_verify.filter(col("prescriber_rank").isNull()).count() == 0, "Null ranks found"
assert df_verify.filter(col("prescriber_rank") < 1).count() == 0, "Invalid ranks found"
assert df_verify.filter(col("total_claims") < 1).count() == 0, "Invalid claims found"
assert df_verify.filter(col("total_cost_usd") < 0).count() == 0, "Negative costs found"
assert df_verify.count() == 23850038, "Row count mismatch"

print("All Gold DQ checks passed ✅")

All Gold DQ checks passed ✅
